# MVPruner full finetune on Kaggle
Full FP32 two-GPU training. This notebook never runs align. With no finetune checkpoint it starts directly from the pretrained T5 and newly initialized multimodal modules.

In [ ]:
import os, json
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/huyvanzzz/Group_HND_car.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'huy-router')
CACHE_HF_REPO_ID = ''  # Optional dataset repo containing cache/*
CACHE_HF_REPO_TYPE = 'dataset'
FINETUNE_CHECKPOINT_HF_REPO_ID = ''  # Optional model repo containing checkpoint + .rng sidecar
FINETUNE_CHECKPOINT_HF_REPO_TYPE = 'model'
FINETUNE_CHECKPOINT_FILENAME = 'finetune_latest.pt'
os.environ.update({'GITHUB_REPO_URL': GITHUB_REPO_URL, 'GITHUB_BRANCH': GITHUB_BRANCH, 'CONFIG': 'configs/repvit_t5_efficient_mini_kaggle_2gpu_mvpruner.yaml', 'PROFILE': 'outputs/repvit_t5_efficient_mini_kaggle_2gpu_mvpruner', 'CACHE_HF_REPO_ID': CACHE_HF_REPO_ID, 'CACHE_HF_REPO_TYPE': CACHE_HF_REPO_TYPE})
os.environ['FINETUNE_CHECKPOINT_HF_REPO_ID'] = FINETUNE_CHECKPOINT_HF_REPO_ID
os.environ['FINETUNE_CHECKPOINT_HF_REPO_TYPE'] = FINETUNE_CHECKPOINT_HF_REPO_TYPE
os.environ['FINETUNE_CHECKPOINT_FILENAME'] = FINETUNE_CHECKPOINT_FILENAME
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception as exc:
    print('HF_TOKEN was not loaded:', exc)
print(json.dumps({'repo': GITHUB_REPO_URL, 'branch': GITHUB_BRANCH, 'cache_hf_repo': CACHE_HF_REPO_ID, 'finetune_checkpoint_hf_repo': FINETUNE_CHECKPOINT_HF_REPO_ID, 'finetune_checkpoint_filename': FINETUNE_CHECKPOINT_FILENAME, 'hf_token_present': bool(os.environ.get('HF_TOKEN'))}, indent=2))

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
python -m pip install -q -e ".[mvpruner]"
python -m pip install -q pycocoevalcap
python - <<'PY'
import efficient_vlm_ad, os, torch, tokenizers, transformers
from efficient_vlm_ad.config import load_config
assert transformers.__version__ == "4.46.3", transformers.__version__
assert tokenizers.__version__ == "0.20.3", tokenizers.__version__
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'
assert torch.cuda.device_count() >= 2, 'Select a Kaggle 2-GPU accelerator'
cfg = load_config(os.environ['CONFIG'])
assert cfg.runtime.precision == 'fp32'
assert cfg.model.fusion.name == 'mvpruner_two_stage'
print('package', efficient_vlm_ad.__file__)
print('torch', torch.__version__, 'transformers', transformers.__version__, 'tokenizers', tokenizers.__version__, 'precision', cfg.runtime.precision)
print('gpus', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
PY
python -m efficient_vlm_ad --help >/dev/null
nvidia-smi

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
mkdir -p "$PROFILE"
if [ -n "$CACHE_HF_REPO_ID" ] && [ ! -f "$PROFILE/cache/manifest.json" ]; then
  hf download "$CACHE_HF_REPO_ID" --repo-type "$CACHE_HF_REPO_TYPE" --local-dir "$PROFILE" --include 'cache/*'
fi
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full
if [ -f "$PROFILE/cache/features.float16.memmap" ] && [ -f "$PROFILE/cache/manifest.json" ]; then
  echo 'Using existing full feature cache.'
else
  python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full
fi
find "$PROFILE" -maxdepth 3 -type f | head -n 30

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage finetune --debug --debug-samples 1 --debug-numerics

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
mkdir -p "$PROFILE/checkpoints"
RESUME_PATH=""
LOCAL_LATEST="$PROFILE/checkpoints/finetune_latest.pt"
if [ -f "$LOCAL_LATEST" ]; then
  echo "Using local finetune checkpoint: $LOCAL_LATEST"
  RESUME_PATH="$LOCAL_LATEST"
elif [ -n "$FINETUNE_CHECKPOINT_HF_REPO_ID" ]; then
  echo "Restoring finetune checkpoint and RNG sidecars from Hugging Face: $FINETUNE_CHECKPOINT_HF_REPO_ID"
  hf download "$FINETUNE_CHECKPOINT_HF_REPO_ID" --repo-type "$FINETUNE_CHECKPOINT_HF_REPO_TYPE" --local-dir "$PROFILE/checkpoints" --include "$FINETUNE_CHECKPOINT_FILENAME" "$FINETUNE_CHECKPOINT_FILENAME.rng/**"
  RESUME_PATH="$PROFILE/checkpoints/$FINETUNE_CHECKPOINT_FILENAME"
  if [ ! -f "$RESUME_PATH" ]; then
    echo "Downloaded checkpoint is missing: $RESUME_PATH" >&2
    exit 1
  fi
else
  echo 'No finetune checkpoint configured; starting finetune directly from pretrained weights.'
fi
RESUME=""
if [ -n "$RESUME_PATH" ]; then
  if [ ! -f "$RESUME_PATH.rng/manifest.json" ]; then
    echo "Distributed RNG manifest is missing: $RESUME_PATH.rng/manifest.json" >&2
    exit 1
  fi
  python -m efficient_vlm_ad verify-resume-checkpoint --config "$CONFIG" --stage finetune --checkpoint "$RESUME_PATH" --debug --debug-samples 1 --debug-numerics
  RESUME="--resume $RESUME_PATH"
fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune $RESUME --debug --debug-samples 1 --debug-numerics
tail -n 30 "$PROFILE/debug/train_progress.jsonl" || true
ls -lh "$PROFILE/checkpoints"

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
if [ ! -f "$CHECKPOINT" ]; then CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"; fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" --batch-size 16
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT"
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/generation_diagnostics.json" || true
cat "$PROFILE/benchmark.json" || true
head -n 20 "$PROFILE/predictions.jsonl" || true

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
ARCHIVE=/kaggle/working/mvpruner_finetune_artifacts.tar.gz
tar -czf "$ARCHIVE" "$PROFILE/checkpoints" "$PROFILE/metrics.json" "$PROFILE/benchmark.json" "$PROFILE/predictions.jsonl" "$PROFILE/debug"
ls -lh "$ARCHIVE"